# Westminster hackathon — TEAM `2CC9AA36` — reproducible notebook

**Vazifa:** har bir alert (`signal_id`) uchun eskalatsiya (`eskalatsiya = 1`) ehtimolini bashorat qilish. Metrika — ROC-AUC.

**Yakuniy yechim:** baseline-65 feature → 3 model (LR, LightGBM, CatBoost), bir xil 5-fold × 3 takror CV →
fold ichida rank-average **50 / 25 / 25** (vaznlar oldindan belgilangan) → LR test taqsimotiga quantile mapping.

| Model | CV AUC (15 fold, mean) | Loyihada qayd etilgan |
|---|---|---|
| Logistic regression (baseline 65) | 0.6453 | ± 0.0116 (ddof=1) |
| LightGBM | 0.6400 | ± 0.0118 (ddof=1) / 0.0114 (ddof=0) |
| CatBoost | 0.6391 | ± 0.0108 (ddof=1) / 0.0105 (ddof=0) |
| **Blend LR+LGB+CAT 50/25/25** | **0.6478** | ± 0.0105 (ddof=1) / 0.0101 (ddof=0) |

Notebook **mustaqil**: loyihaning `src/` papkasiga bog'liq emas — anchor qoidasi va feature kodi shu yerda.
U **xom** ma'lumotdan (`data/raw/`, parquet → pyarrow) boshlab hammasini qayta hisoblaydi. Yo'llar nisbiy, `SEED = 42`.

**Colab'da ishga tushirish** (batafsil — oxirgi bo'limda):
1. Notebook bilan bir papkada `data/raw/` yarating va yuklang: `train_signals.csv`, `test_signals.csv`, `sample_submission.csv`,
   tranzaksiyalar — asl bo'laklar `data/raw/parts/{train,test}_transactions_part*.parquet` (tavsiya) **yoki** to'liq
   `data/raw/{train,test}_transactions.parquet`.
2. Ixtiyoriy: `outputs/folds.csv` (bo'lmasa seed=42 bilan tiklanadi) va `outputs/team_2CC9AA36.csv` (oxirgi solishtirish uchun).
3. *Runtime → Run all.* Taxminiy vaqt (Colab CPU): ~10–20 daqiqa (asosan CatBoost).

Notebook yakuniy faylni **ustidan yozmaydi**: natija `outputs/notebook/notebook_output.csv` ga yoziladi.

## 1. Sozlamalar
Kutubxonalar, `SEED`, nisbiy yo'llar va ikki bayroq:
- `LR_TEST_MODE = "full_fit"` (**standart**) — LR test bashorati butun train'da o'qitilgan bitta LR dan; loyihadagi yakuniy fayl
  aynan shu bilan olingan, shuning uchun notebook o'sha faylni bit darajasida qayta hosil qiladi.
  `"fold_mean"` — muqobil: LR test bashorati 15 fold-modelning o'rtachasi (LGB/CAT bilan bir xil sxema).
- `GBM_SOURCE = "train"` — LightGBM/CatBoost shu yerda o'qitiladi (Colab). `"saved"` — oldingi Colab natijalari
  (`data/colab/{oof,test}_{lgb,cat}.csv`) o'qiladi (masalan, lightgbm/catboost o'rnatib bo'lmaydigan muhitda).

In [1]:
# Colab: lightgbm va pyarrow odatda o'rnatilgan, catboost esa yo'q
!pip -q install catboost

import json, os, re, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

RAW = Path("data/raw")                 # xom ma'lumot
PARTS = RAW / "parts"                  # asl parquet bo'laklari (ixtiyoriy)
FOLDS_CSV = Path("outputs/folds.csv")  # ixtiyoriy: loyihadagi foldlar
FINAL_CSV = Path("outputs/team_2CC9AA36.csv")   # ixtiyoriy: loyihadagi yakuniy fayl (faqat solishtirish)
SAVED_GBM = Path("data/colab")         # GBM_SOURCE="saved" uchun
OUT = Path("outputs/notebook")         # notebook chiqishlari (yakuniy faylga tegmaydi)
OUT.mkdir(parents=True, exist_ok=True)

LR_TEST_MODE = "full_fit"    # "full_fit" (standart, yakuniy fayl) | "fold_mean" (muqobil)
GBM_SOURCE = "train"         # "train" | "saved"
TARGET, ID, PRED = "eskalatsiya", "signal_id", "ehtimollik"
T0 = time.time()
import sklearn, scipy
print("pandas", pd.__version__, "| numpy", np.__version__, "| sklearn", sklearn.__version__, "| scipy", scipy.__version__)

   ━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/97.1 MB 134.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━ 63.1/97.1 MB 150.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━ 91.0/97.1 MB 192.6 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 159.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 159.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 159.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 159.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 159.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.4 MB/s eta 0:00:00


pandas 2.2.3 | numpy 2.1.3 | sklearn 1.6.1 | scipy 1.16.3


## 2. Ma'lumot yuklash
Signal jadvallari (CSV) va tranzaksiyalar (parquet, pyarrow). Agar `data/raw/parts/` mavjud bo'lsa, **asl bo'laklar** ishlatiladi
(ular pyarrow bilan yozilgan asl fayllar; tartib part1…partN — loyihadagi birlashtirish bilan bir xil). Aks holda to'liq fayllar.
Qator sonlari loyihada qayd etilganlar bilan tekshiriladi.

In [2]:
TX_COLS = ["signal_id", "tranzaksiya_vaqti", "kirim_chiqim", "tranzaksiya_turi", "miqdor_indeksi"]


def read_parquet(path):
    return pd.read_parquet(path, columns=TX_COLS)


def load_tx(split):
    parts = sorted(PARTS.glob(f"{split}_transactions_part*.parquet"),
                   key=lambda p: int(re.findall(r"(\d+)", p.stem)[-1])) if PARTS.exists() else []   # part1 / part_1 / part-1
    if parts:
        src = [str(p) for p in parts]
        df = pd.concat([read_parquet(p) for p in parts], ignore_index=True)
    else:
        src = [str(RAW / f"{split}_transactions.parquet")]
        df = read_parquet(src[0])
    df["tranzaksiya_vaqti"] = pd.to_datetime(df["tranzaksiya_vaqti"])
    df["miqdor_indeksi"] = df["miqdor_indeksi"].astype("float64")
    print(f"{split}: {len(df):,} tx  <- {src}")
    return df


train_sig = pd.read_csv(RAW / "train_signals.csv", parse_dates=["signal_sanasi"])
test_sig = pd.read_csv(RAW / "test_signals.csv", parse_dates=["signal_sanasi"])
sample_sub = pd.read_csv(RAW / "sample_submission.csv")
train_tx, test_tx = load_tx("train"), load_tx("test")

assert len(train_sig) == 14000 and len(test_sig) == 6000
assert len(train_tx) == 6_987_663 and len(test_tx) == 3_027_575, "tranzaksiyalar soni loyihadagidan farq qiladi"
assert set(train_tx.signal_id) == set(train_sig.signal_id) and set(test_tx.signal_id) == set(test_sig.signal_id)
print("eskalatsiya ulushi:", round(train_sig[TARGET].mean(), 4))

train: 6,987,663 tx  <- ['data/raw/parts/train_transactions_part1.parquet', 'data/raw/parts/train_transactions_part2.parquet', 'data/raw/parts/train_transactions_part3.parquet', 'data/raw/parts/train_transactions_part4.parquet', 'data/raw/parts/train_transactions_part5.parquet']


test: 3,027,575 tx  <- ['data/raw/parts/test_transactions_part1.parquet', 'data/raw/parts/test_transactions_part2.parquet', 'data/raw/parts/test_transactions_part3.parquet']


eskalatsiya ulushi: 0.1718


## 3. Anchor qoidasi
Har bir signal uchun "hozir" nuqtasi: **`anchor = max(signal_sanasi 00:00, oxirgi tranzaksiya kunining oxiri)`**.
99.9% signalda tarix aynan `signal_sanasi 00:00` da yopiladi; faqat 2025-02-28 dagi 23 signalda (train 21, test 2)
tranzaksiyalar shu kun 23:59 gacha davom etadi (ma'lumot artefakti) — ular uchun anchor = +1 kun.
Anchor dan keyin hech qanday tranzaksiya yo'q (tekshiriladi), shuning uchun kelajakdagi ma'lumot featurelarga kirmaydi.

In [3]:
def compute_anchor(signals: pd.DataFrame, tx: pd.DataFrame) -> pd.Series:
    '''signal_id -> anchor. anchor = max(signal_sanasi 00:00, ceil_day(oxirgi tx)).'''
    last = tx.groupby("signal_id", observed=True)["tranzaksiya_vaqti"].max()
    s = signals.set_index("signal_id")["signal_sanasi"]
    last = last.reindex(s.index)
    ceil_last = last.dt.ceil("D")
    anchor = s.where(ceil_last.isna() | (ceil_last <= s), ceil_last)
    return anchor.rename("anchor")


# train + test birga (bitta funksiya, target ishlatilmaydi)
all_sig = pd.concat([train_sig[["signal_id", "signal_sanasi"]], test_sig[["signal_id", "signal_sanasi"]]], ignore_index=True)
all_tx = pd.concat([train_tx, test_tx], ignore_index=True)
del train_tx, test_tx
anchor = compute_anchor(all_sig, all_tx)
shifted = anchor != all_sig.set_index("signal_id")["signal_sanasi"]
after = int((all_tx["tranzaksiya_vaqti"].to_numpy() > all_tx["signal_id"].map(anchor).to_numpy()).sum())
print("anchor surilgan signallar:", int(shifted.sum()), "| sanalar:", sorted({str(d.date()) for d in all_sig.set_index('signal_id').signal_sanasi[shifted]}))
print("anchor dan keyingi tx:", after)
assert int(shifted.sum()) == 23 and after == 0

anchor surilgan signallar: 23 | sanalar: ['2025-02-28']
anchor dan keyingi tx: 0


## 4. Baseline-65 featurelar
Bitta funksiya (`build_baseline_features`) train va test uchun **bir xil** qo'llanadi; target ishlatilmaydi.
Guruhlar (loyiha EDA va ablation natijasi — signal asosan tur × yo'nalish bo'yicha **miqdor darajasida**):
- **a (27):** hajm, kirim/tur ulushlari, umumiy miqdor statistikalari, tarix uzunligi, soat/dam olish, tur va yo'nalish bo'yicha o'rtacha miqdor;
- **t (32):** tur bo'yicha miqdor min/max/std/median/q10/q90 va tur × yo'nalish o'rtachasi;
- **f (6):** "floor" — miqdorning global minimumi (−2.912314, faqat bank o'tkazmasi kirimida uchraydi) atrofidagi tranzaksiyalar.

65 ta ustun ro'yxati va tartibi loyihadagi `feature_groups_v1.json["base"]` bilan aynan bir xil (notebook ichiga qo'yilgan).

In [4]:
BASE65 = ['a_n', 'a_kir_sh', 'a_t_karta_sh', 'a_t_bank_sh', 'a_t_naqd_sh', 'a_t_xalq_sh', 'a_amt_sum', 'a_amt_mean',
          'a_amt_max', 'a_amt_std', 'a_money_sum', 'a_out_in_ratio', 'a_hist_days', 'a_tx_per_day', 'a_amt_min',
          'a_amt_q10', 'a_amt_q50', 'a_amt_q90', 'a_hour_mean', 'a_night_sh', 'a_weekend_sh', 'a_amt_mean_karta',
          'a_amt_mean_bank', 'a_amt_mean_naqd', 'a_amt_mean_xalq', 'a_amt_mean_kirim', 'a_amt_mean_chiqim',
          't_amt_min_karta', 't_amt_min_bank', 't_amt_min_naqd', 't_amt_min_xalq', 't_amt_max_karta', 't_amt_max_bank',
          't_amt_max_naqd', 't_amt_max_xalq', 't_amt_std_karta', 't_amt_std_bank', 't_amt_std_naqd', 't_amt_std_xalq',
          't_amt_median_karta', 't_amt_median_bank', 't_amt_median_naqd', 't_amt_median_xalq', 't_amt_q10_karta',
          't_amt_q10_bank', 't_amt_q10_naqd', 't_amt_q10_xalq', 't_amt_q90_karta', 't_amt_q90_bank', 't_amt_q90_naqd',
          't_amt_q90_xalq', 't_amt_mean_karta_kirim', 't_amt_mean_karta_chiqim', 't_amt_mean_bank_kirim',
          't_amt_mean_bank_chiqim', 't_amt_mean_naqd_kirim', 't_amt_mean_naqd_chiqim', 't_amt_mean_xalq_kirim',
          't_amt_mean_xalq_chiqim', 'f_n_floor', 'f_floor_flag', 'f_min_minus_floor', 'f_bank_kirim_min',
          'f_bank_kirim_n', 'f_floor_share_bank_kirim']
assert len(BASE65) == 65

TYPES = ["karta", "bank_otkazmasi", "naqd", "xalqaro"]
TS = ["karta", "bank", "naqd", "xalq"]
FLOOR = -2.912314   # train tranzaksiyalaridagi global min miqdor_indeksi
FLOOR_BAND = 0.01


def build_baseline_features(signals: pd.DataFrame, tx: pd.DataFrame) -> pd.DataFrame:
    '''signals: signal_id, signal_sanasi; tx: tranzaksiyalar. Qaytaradi: X (index=signal_id, 65 ustun).'''
    ids = signals["signal_id"].to_numpy()
    anchor = compute_anchor(signals, tx)
    d = pd.DataFrame({
        "sid": pd.Categorical(tx["signal_id"], categories=ids).codes.astype(np.int32),
        "kir": (tx["kirim_chiqim"] == "kirim").to_numpy().astype(np.int8),
        "typ": pd.Categorical(tx["tranzaksiya_turi"], categories=TYPES).codes.astype(np.int8),
        "amt": tx["miqdor_indeksi"].to_numpy(np.float64),
    })
    t = tx["tranzaksiya_vaqti"]
    d["h"] = (pd.Series(anchor.to_numpy())[d["sid"]].to_numpy() - t.to_numpy()) / np.timedelta64(1, "h")
    d["hour"] = t.dt.hour.to_numpy()
    d["wday"] = t.dt.dayofweek.to_numpy()
    d["money"] = np.exp(d["amt"])
    for i, s in enumerate(TS):
        d[f"t_{s}"] = (d["typ"] == i).astype(np.int8)
    N = len(ids)

    def full(s, fill=np.nan):
        return s.reindex(range(N)).fillna(fill) if fill is not None else s.reindex(range(N))

    g = d.groupby("sid")
    X = pd.DataFrame(index=range(N))
    # (a) umumiy
    X["a_n"] = full(g.size(), 0)
    m = g[["kir", "t_karta", "t_bank", "t_naqd", "t_xalq"]].mean()
    for c in m.columns:
        X[f"a_{c}_sh"] = full(m[c])
    a = g["amt"].agg(["sum", "mean", "max", "std"])
    for c in a.columns:
        X[f"a_amt_{c}"] = full(a[c], 0 if c == "sum" else np.nan)
    X["a_money_sum"] = full(g["money"].sum(), 0)
    dk = d.assign(mi=d["money"] * d["kir"], mo=d["money"] * (1 - d["kir"])).groupby("sid")[["mi", "mo"]].sum()
    X["a_out_in_ratio"] = full(np.log1p(dk["mo"]) - np.log1p(dk["mi"]))
    X["a_hist_days"] = full(g["h"].max() / 24)
    X["a_tx_per_day"] = X["a_n"] / X["a_hist_days"].clip(lower=1)
    X["a_amt_min"] = full(g["amt"].min())
    for q in (0.1, 0.5, 0.9):
        X[f"a_amt_q{int(q * 100)}"] = full(g["amt"].quantile(q))
    X["a_hour_mean"] = full(g["hour"].mean())
    X["a_night_sh"] = full((d["hour"] < 6).groupby(d["sid"]).mean())
    X["a_weekend_sh"] = full((d["wday"] >= 5).groupby(d["sid"]).mean())
    gt = d.groupby(["sid", "typ"])["amt"]
    pt = gt.mean().unstack()
    for i, s in enumerate(TS):
        X[f"a_amt_mean_{s}"] = full(pt[i]) if i in pt else np.nan
    pdir = d.groupby(["sid", "kir"])["amt"].mean().unstack()
    X["a_amt_mean_kirim"], X["a_amt_mean_chiqim"] = full(pdir[1]), full(pdir[0])
    # (t) tur bo'yicha miqdor
    st = {"min": gt.min(), "max": gt.max(), "std": gt.std(), "median": gt.median(),
          "q10": gt.quantile(0.1), "q90": gt.quantile(0.9)}
    for nm, ser in st.items():
        u = ser.unstack()
        for i, s in enumerate(TS):
            X[f"t_amt_{nm}_{s}"] = full(u[i]) if i in u else np.nan
    ptd = d.groupby(["sid", "typ", "kir"])["amt"].mean().unstack([1, 2])
    for i, s in enumerate(TS):
        for k, kn in ((1, "kirim"), (0, "chiqim")):
            X[f"t_amt_mean_{s}_{kn}"] = full(ptd[(i, k)]) if (i, k) in ptd else np.nan
    # (f) floor
    band = d["amt"] <= FLOOR + FLOOR_BAND
    X["f_n_floor"] = full(band.groupby(d["sid"]).sum(), 0)
    X["f_floor_flag"] = (X["f_n_floor"] > 0).astype(np.int8)
    X["f_min_minus_floor"] = X["a_amt_min"] - FLOOR
    bk = d[(d["typ"] == 1) & (d["kir"] == 1)].groupby("sid")["amt"]
    X["f_bank_kirim_min"] = full(bk.min())
    X["f_bank_kirim_n"] = full(bk.size(), 0)
    X["f_floor_share_bank_kirim"] = X["f_n_floor"] / X["f_bank_kirim_n"].replace(0, np.nan)
    X.index = pd.Index(ids, name="signal_id")
    return X[BASE65].replace([np.inf, -np.inf], np.nan).astype(np.float64)


X_all = build_baseline_features(all_sig, all_tx)
del all_tx
Xtr = X_all.loc[train_sig.signal_id]
Xte = X_all.loc[test_sig.signal_id]
y = train_sig.set_index("signal_id")[TARGET].astype(int).loc[Xtr.index]
assert list(Xtr.columns) == BASE65 and Xtr.shape == (14000, 65) and Xte.shape == (6000, 65)
print("Xtr", Xtr.shape, "Xte", Xte.shape, "| NaN ulushi train/test:",
      round(float(Xtr.isna().to_numpy().mean()), 4), round(float(Xte.isna().to_numpy().mean()), 4),
      f"| {time.time() - T0:.0f}s")

# ixtiyoriy: loyihadagi saqlangan featurelar bilan solishtirish (fayl bo'lsa)
_saved = Path("data/features/features_v1_train.csv.gz")
if _saved.exists():
    s_tr = pd.read_csv(_saved, index_col="signal_id", float_precision="round_trip")[BASE65]
    a_, b_ = Xtr.to_numpy(), s_tr.loc[Xtr.index].to_numpy()
    same_nan = bool((np.isnan(a_) == np.isnan(b_)).all()); mm = ~np.isnan(a_)
    print("saqlangan features_v1 bilan: NaN maskasi teng:", same_nan, "| max abs farq:", float(np.max(np.abs(a_[mm] - b_[mm]))))

Xtr (14000, 65) Xte (6000, 65) | NaN ulushi train/test: 0.0553 0.0538 | 21s


## 5. CV foldlari
5 fold × 3 takror `RepeatedStratifiedKFold(random_state=42)` train_signals tartibida. Agar loyihadagi `outputs/folds.csv`
yuklangan bo'lsa — o'qiladi va seed dan tiklangan foldlar bilan **aynan bir xil** ekanligi tekshiriladi.
Hamma model shu 15 fold ustida baholanadi — shuning uchun fold bo'yicha juft taqqoslash va blend to'g'ri.

In [5]:
def make_folds(yv, ids):
    rows = []
    rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
    for k, (_, va) in enumerate(rskf.split(np.zeros(len(yv)), yv)):
        rows.append(pd.DataFrame({"signal_id": ids[va], "repeat": k // 5, "fold": k % 5}))
    return pd.concat(rows, ignore_index=True).sort_values(["repeat", "signal_id"]).reset_index(drop=True)


folds_seed = make_folds(train_sig[TARGET].to_numpy(), train_sig["signal_id"].to_numpy())
if FOLDS_CSV.exists():
    folds = pd.read_csv(FOLDS_CSV).sort_values(["repeat", "signal_id"]).reset_index(drop=True)
    print("folds.csv == seed=42 bilan tiklangan:", folds.equals(folds_seed))
    assert folds.equals(folds_seed)
else:
    folds = folds_seed
    print("folds.csv yo'q — seed=42 bilan tiklandi")
print(folds.groupby(["repeat", "fold"]).size().unstack())


def fold_ids(r):
    return folds[folds.repeat == r].set_index("signal_id").fold.reindex(y.index).to_numpy()


def summarize(name, aucs):
    aucs = np.asarray(aucs)
    print(f"{name}: CV AUC {aucs.mean():.4f} ± {aucs.std(ddof=1):.4f} (ddof=1) / ± {aucs.std():.4f} (ddof=0) | "
          f"takrorlar {[round(float(aucs[i*5:(i+1)*5].mean()), 4) for i in range(3)]}")

folds.csv yo'q — seed=42 bilan tiklandi
fold       0     1     2     3     4
repeat                              
0       2800  2800  2800  2800  2800
1       2800  2800  2800  2800  2800
2       2800  2800  2800  2800  2800


## 6. Logistic regression (asosiy model)
`median imputatsiya → StandardScaler → LogisticRegressionCV` (L2, C panjarasi 1e-4…0.3, ichki 3-fold AUC, SEED=42).
Imputer va scaler har foldda **faqat train qismidan** fit qilinadi. Loyihada ichki CV deyarli hamma joyda **C = 0.01** ni tanlaydi
(butun train'da ham). EDA: bog'liqlik chiziqli-additiv — LR daraxtli modellardan kuchliroq.
- OOF: 15 fold-model (har takror uchun 5).
- Test: `LR_TEST_MODE="full_fit"` (**standart**) → butun train'dagi bitta model (loyihadagi yakuniy fayl shu bilan olingan);
  `"fold_mean"` (muqobil) → 15 fold-model o'rtachasi. Ikkalasi ham hisoblanadi, tanlangani blend'ga ketadi.

In [6]:
CS = [1e-4, 3e-4, 1e-3, 3e-3, 0.01, 0.03, 0.1, 0.3]


def lr_model():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegressionCV(Cs=CS, cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
                                              scoring="roc_auc", max_iter=4000, n_jobs=-1))


Xtr_np, Xte_np, y_np = Xtr.to_numpy(np.float64), Xte.to_numpy(np.float64), y.to_numpy()
oof_lr = np.zeros((3, len(y))); test_lr_sum = np.zeros(len(Xte)); a_lr = []; chosen_C = []
for r in range(3):
    f = fold_ids(r)
    for k in range(5):
        tri, vai = np.where(f != k)[0], np.where(f == k)[0]
        m = lr_model().fit(Xtr_np[tri], y_np[tri])
        oof_lr[r, vai] = m.predict_proba(Xtr_np[vai])[:, 1]
        test_lr_sum += m.predict_proba(Xte_np)[:, 1]
        a_lr.append(roc_auc_score(y_np[vai], oof_lr[r, vai])); chosen_C.append(float(m[-1].C_[0]))
a_lr = np.array(a_lr)
summarize("LR", a_lr)
print("foldlarda tanlangan C:", pd.Series(chosen_C).value_counts().to_dict())
test_lr_foldmean = test_lr_sum / 15

lr_full = lr_model().fit(Xtr_np, y_np)
test_lr_full = lr_full.predict_proba(Xte_np)[:, 1]
print("butun train'da tanlangan C:", float(lr_full[-1].C_[0]),
      "| Spearman(fold_mean, full_fit) test:", round(float(stats.spearmanr(test_lr_foldmean, test_lr_full)[0]), 5))
test_lr = test_lr_foldmean if LR_TEST_MODE == "fold_mean" else test_lr_full
print("LR_TEST_MODE =", LR_TEST_MODE, f"| {time.time() - T0:.0f}s")

LR: CV AUC 0.6453 ± 0.0116 (ddof=1) / ± 0.0112 (ddof=0) | takrorlar [0.6453, 0.6463, 0.6444]
foldlarda tanlangan C: {0.01: 12, 0.03: 1, 0.3: 1, 0.1: 1}


butun train'da tanlangan C: 0.01 | Spearman(fold_mean, full_fit) test: 0.99975
LR_TEST_MODE = full_fit | 39s


## 7. LightGBM va CatBoost
Colab'da LGB/CAT ni o'qitgan **aynan o'sha kod** (parametrlar va seedlar o'zgarmagan):
har fold modeli seed = `SEED + 10*r + k`, early stopping uchun train qismidan **15% stratified** ichki ajratma (`random_state=seed`).
Kuchli regularizatsiya (depth 3, katta L2) — EDA bo'yicha signal kuchsiz va asosan chiziqli. Test bashorati = 15 fold-model o'rtachasi.
`run_cv` asl ko'rinishida (u std ni `np.std`, ya'ni ddof=0 bilan chiqaradi); fayllar `outputs/notebook/` ga yoziladi.
`GBM_SOURCE="saved"` bo'lsa, o'qitish o'rniga oldingi Colab natijalari o'qiladi.

In [7]:
def run_cv(fit_predict, name):
    oof = np.zeros((3, len(y))); test_sum = np.zeros(len(Xte)); aucs = []; nfit = 0
    for r in range(3):
        f = fold_ids(r)
        for k in range(5):
            tri, vai = np.where(f != k)[0], np.where(f == k)[0]
            pv, pt = fit_predict(Xtr.iloc[tri], y.iloc[tri], Xtr.iloc[vai], Xte, SEED + 10*r + k)
            oof[r, vai] = pv; test_sum += pt; nfit += 1
            aucs.append(roc_auc_score(y.iloc[vai], pv))
    aucs = np.array(aucs)
    print(f"{name}: CV AUC {aucs.mean():.4f} ± {aucs.std():.4f}  (15 fold)")
    pd.DataFrame({f"oof_r{r}": oof[r] for r in range(3)}, index=y.index).assign(oof_mean=oof.mean(0)).to_csv(OUT / f"oof_{name}.csv")
    pd.Series(test_sum / nfit, index=Xte.index, name="pred").to_csv(OUT / f"test_{name}.csv")
    return oof, test_sum / nfit, aucs


if GBM_SOURCE == "train":
    import lightgbm as lgb
    from catboost import CatBoostClassifier

    def fp_lgb(Xa, ya, Xv, Xt, seed):
        Xf, Xe, yf, ye = train_test_split(Xa, ya, test_size=0.15, stratify=ya, random_state=seed)
        m = lgb.LGBMClassifier(n_estimators=3000, learning_rate=0.02, max_depth=3, num_leaves=6,
            min_child_samples=100, subsample=0.7, subsample_freq=1, colsample_bytree=0.5,
            reg_lambda=20, random_state=seed, verbose=-1)
        m.fit(Xf, yf, eval_set=[(Xe, ye)], eval_metric="auc",
              callbacks=[lgb.early_stopping(100, verbose=False)])
        return m.predict_proba(Xv)[:, 1], m.predict_proba(Xt)[:, 1]

    def fp_cat(Xa, ya, Xv, Xt, seed):
        Xf, Xe, yf, ye = train_test_split(Xa, ya, test_size=0.15, stratify=ya, random_state=seed)
        m = CatBoostClassifier(iterations=3000, learning_rate=0.03, depth=3, l2_leaf_reg=20,
            eval_metric="AUC", random_seed=seed, verbose=0, od_type="Iter", od_wait=100, thread_count=-1)
        m.fit(Xf, yf, eval_set=(Xe, ye), use_best_model=True)
        return m.predict_proba(Xv)[:, 1], m.predict_proba(Xt)[:, 1]

    print("lightgbm", lgb.__version__)
    oof_lgb, test_lgb, a_lgb = run_cv(fp_lgb, "lgb")
    oof_cat, test_cat, a_cat = run_cv(fp_cat, "cat")
else:
    def load_saved(name):
        o = pd.read_csv(SAVED_GBM / f"oof_{name}.csv", index_col=0, float_precision="round_trip").loc[y.index]
        t = pd.read_csv(SAVED_GBM / f"test_{name}.csv", index_col=0, float_precision="round_trip").loc[Xte.index, "pred"]
        oof = np.vstack([o[f"oof_r{r}"].to_numpy() for r in range(3)])
        aucs = np.array([roc_auc_score(y_np[fold_ids(r) == k], oof[r][fold_ids(r) == k]) for r in range(3) for k in range(5)])
        return oof, t.to_numpy(), aucs
    oof_lgb, test_lgb, a_lgb = load_saved("lgb")
    oof_cat, test_cat, a_cat = load_saved("cat")
    print("GBM_SOURCE = saved (data/colab dan o'qildi)")
summarize("LGB", a_lgb)
summarize("CAT", a_cat)
print(f"{time.time() - T0:.0f}s")

lightgbm 4.6.0


lgb: CV AUC 0.6400 ± 0.0114  (15 fold)


cat: CV AUC 0.6391 ± 0.0105  (15 fold)
LGB: CV AUC 0.6400 ± 0.0118 (ddof=1) / ± 0.0114 (ddof=0) | takrorlar [0.6405, 0.6385, 0.641]
CAT: CV AUC 0.6391 ± 0.0108 (ddof=1) / ± 0.0105 (ddof=0) | takrorlar [0.6393, 0.6386, 0.6393]
125s


## 8. Blend: fold ichida rank-average 50 / 25 / 25
Har bir fold validatsiyasida bashoratlar shu fold ichida rank'larga (0..1) aylantiriladi va `0.50·LR + 0.25·LGB + 0.25·CAT`
bilan qo'shiladi. Vaznlar **oldindan belgilangan** (OOF da optimallashtirilmagan) — shuning uchun tanlov optimizmi yo'q.
LR ga nisbatan juft fold farqi ham chiqariladi (loyihadagi qat'iy mezon: ≥ 0.003 va ≥ 12/15 fold — blend bu mezondan
o'tmagan, lekin 3 takrorda ham musbat).

In [8]:
W = {"lr": 0.50, "lgb": 0.25, "cat": 0.25}
oofs = {"lr": oof_lr, "lgb": oof_lgb, "cat": oof_cat}


def blend_aucs(w):
    res = []
    for r in range(3):
        f = fold_ids(r)
        for k in range(5):
            m = f == k
            s = sum(wt * stats.rankdata(oofs[n][r][m]) / m.sum() for n, wt in w.items())
            res.append(roc_auc_score(y.values[m], s))
    return np.array(res)


a_blend = blend_aucs(W)
summarize("BLEND 50/25/25", a_blend)
d = a_blend - a_lr
print(f"Blend − LR: {d.mean():+.5f} ± {d.std(ddof=1):.4f} | yaxshi {int((d > 0).sum())}/15 | "
      f"takrorlar {[round(float(d[i*5:(i+1)*5].mean()), 4) for i in range(3)]}")

expected = {"LR": 0.6453, "LGB": 0.6400, "CAT": 0.6391, "BLEND": 0.6478}
got = {"LR": a_lr.mean(), "LGB": a_lgb.mean(), "CAT": a_cat.mean(), "BLEND": a_blend.mean()}
print(pd.DataFrame({"kutilgan": expected, "olingan": {k: round(float(v), 4) for k, v in got.items()},
                    "farq": {k: round(float(got[k] - expected[k]), 4) for k in expected}}))

BLEND 50/25/25: CV AUC 0.6478 ± 0.0105 (ddof=1) / ± 0.0101 (ddof=0) | takrorlar [0.6478, 0.6477, 0.6478]
Blend − LR: +0.00245 ± 0.0054 | yaxshi 10/15 | takrorlar [0.0025, 0.0014, 0.0034]
       kutilgan  olingan  farq
LR       0.6453   0.6453   0.0
LGB      0.6400   0.6400   0.0
CAT      0.6391   0.6391  -0.0
BLEND    0.6478   0.6478  -0.0


## 9. Test bashorati: rank-average + quantile mapping
`blend = 0.5·rank(LR_test) + 0.25·rank(LGB_test) + 0.25·rank(CAT_test)`, `rank = rankdata / n`.
Keyin **quantile mapping**: blend tartibi bo'yicha saralangan LR test qiymatlari beriladi
(`final[argsort(blend)] = sort(LR_test)`). Natija: tartib = blend (AUC uchun muhim), qiymatlar taqsimoti = LR (kalibrlangan, 0..1).
Eslatma: blend'da ~450–470 ta teng qiymat bor; ular ichidagi tartib fayl tartibi bo'yicha (stable sort). Yig'indi `Σ w·(rank/n)` shaklida —
aks holda floating-point yaxlitlash tie'larni o'zgartirib, AUC ga ta'sirsiz, lekin qiymat darajasida farq beradi.

In [9]:
n = len(test_lr)
tests = {"lr": test_lr, "lgb": test_lgb, "cat": test_cat}
# yig'indi aynan shu tartib va shaklda (w * rank/n) — floating-point tie tuzilmasi loyihadagi final_ensemble.py bilan bir xil bo'lsin
blend_test = sum(W[k] * (stats.rankdata(tests[k]) / n) for k in ("lr", "lgb", "cat"))
final = np.empty(n)
final[np.argsort(blend_test, kind="stable")] = np.sort(test_lr)

o = np.argsort(blend_test, kind="stable")
viol = int(((np.diff(blend_test[o]) > 0) & (np.diff(final[o]) < 0)).sum())
print("monotonlik buzilishi:", viol, "| Spearman(final, blend):", round(float(stats.spearmanr(final, blend_test)[0]), 8),
      "| blend unikal:", len(np.unique(blend_test)))
assert viol == 0 and np.array_equal(np.sort(final), np.sort(test_lr))
print("test korrelyatsiyasi (Spearman):")
print(pd.DataFrame({"LR": test_lr, "LGB": test_lgb, "CAT": test_cat}).corr(method="spearman").round(3))

monotonlik buzilishi: 0 | Spearman(final, blend): 0.99999999 | blend unikal: 5527
test korrelyatsiyasi (Spearman):
        LR    LGB    CAT
LR   1.000  0.897  0.894
LGB  0.897  1.000  0.980
CAT  0.894  0.980  1.000


## 10. CSV yozish
Natija `outputs/notebook/notebook_output.csv` ga yoziladi (`signal_id,ehtimollik`, test_signals tartibida, indekssiz).
Loyihadagi yakuniy `outputs/team_2CC9AA36.csv` **ustidan yozilmaydi** (assert bilan himoyalangan).

In [10]:
NB_OUT = OUT / "notebook_output.csv"
assert NB_OUT.name != "team_2CC9AA36.csv" and NB_OUT.resolve() != FINAL_CSV.resolve()
sub = pd.DataFrame({ID: test_sig[ID].to_numpy(), PRED: final})
sub.to_csv(NB_OUT, index=False)
print("yozildi:", NB_OUT, sub.shape)

yozildi: outputs/notebook/notebook_output.csv (6000, 2)


## 11. Format tekshiruvi
Ustunlar va tartibi (`sample_submission` bilan), 6000 qator, ID lar test_signals bilan aynan va tartibda, dublikat/NaN/inf yo'q, 0 ≤ p ≤ 1.

In [11]:
chk = pd.read_csv(NB_OUT, float_precision="round_trip")
p = chk[PRED]
checks = {
    "sarlavha 'signal_id,ehtimollik'": open(NB_OUT).readline().strip() == "signal_id,ehtimollik",
    "ustunlar = sample_submission": list(chk.columns) == list(sample_sub.columns),
    "6000 qator": len(chk) == 6000,
    "ID to'plami = test_signals": set(chk[ID]) == set(test_sig[ID]),
    "ID tartibi = test_signals": chk[ID].tolist() == test_sig[ID].tolist(),
    "ID tartibi = sample_submission": chk[ID].tolist() == sample_sub[ID].tolist(),
    "dublikat ID yo'q": not chk[ID].duplicated().any(),
    "NaN yo'q": not chk.isna().any().any(),
    "inf yo'q": bool(np.isfinite(p).all()),
    "0 <= p <= 1": bool(p.between(0, 1).all()),
    "doimiy emas": float(p.std()) > 0,
}
for k, v in checks.items():
    print(f"  [{'OK' if v else 'XATO'}] {k}")
assert all(checks.values())
print(f"OK — {sum(checks.values())}/{len(checks)} | p: min {p.min():.6f} max {p.max():.6f} mean {p.mean():.6f}")

  [OK] sarlavha 'signal_id,ehtimollik'
  [OK] ustunlar = sample_submission
  [OK] 6000 qator
  [OK] ID to'plami = test_signals
  [OK] ID tartibi = test_signals
  [OK] ID tartibi = sample_submission
  [OK] dublikat ID yo'q
  [OK] NaN yo'q
  [OK] inf yo'q
  [OK] 0 <= p <= 1
  [OK] doimiy emas
OK — 11/11 | p: min 0.001149 max 0.598326 mean 0.172884


## 12. Loyihadagi yakuniy fayl bilan solishtirish
Agar `outputs/team_2CC9AA36.csv` yuklangan bo'lsa: Spearman va maksimal absolyut farq.
Kutiladigan natija (loyiha sandbox'ida, loyihadagi LGB/CAT natijalari bilan sinalgan):
- `LR_TEST_MODE="full_fit"` (**standart**) → **aynan bir xil** (Spearman 1.0, max |farq| = 0);
- `"fold_mean"` (muqobil) → Spearman ≈ 0.99993, max |farq| ≈ 0.019, top-10% kesishma ≈ 0.992
  (LR test bashorati 15 fold-model o'rtachasi, shuning uchun qiymatlar biroz boshqacha).
LightGBM/CatBoost Colab'da qayta o'qitilganda kutubxona versiyasiga qarab kichik farqlar bo'lishi mumkin.

In [12]:
if FINAL_CSV.exists():
    ref = pd.read_csv(FINAL_CSV, float_precision="round_trip")
    assert ref[ID].tolist() == sub[ID].tolist()
    a_, b_ = sub[PRED].to_numpy(), ref[PRED].to_numpy()
    print(f"Spearman(notebook, yakuniy) = {stats.spearmanr(a_, b_)[0]:.6f}")
    print(f"max |farq| = {np.max(np.abs(a_ - b_)):.3e} | aynan teng: {np.array_equal(a_, b_)}")
    top = 600
    print(f"top-10% kesishma: {len(set(np.argsort(-a_)[:top]) & set(np.argsort(-b_)[:top])) / top:.3f}")
else:
    print("outputs/team_2CC9AA36.csv topilmadi — solishtirish o'tkazib yuborildi")
print(f"jami vaqt: {time.time() - T0:.0f}s")

outputs/team_2CC9AA36.csv topilmadi — solishtirish o'tkazib yuborildi
jami vaqt: 125s


## 13. Colab yo'riqnomasi va eslatmalar
**Yuklanadigan fayllar** (notebook papkasiga nisbatan):
```
data/raw/train_signals.csv
data/raw/test_signals.csv
data/raw/sample_submission.csv
data/raw/parts/train_transactions_part1..5.parquet   # asl bo'laklar (tavsiya etiladi)
data/raw/parts/test_transactions_part1..3.parquet
# yoki bo'laklar o'rniga: data/raw/train_transactions.parquet, data/raw/test_transactions.parquet
outputs/folds.csv                 # ixtiyoriy (tekshiruv uchun)
outputs/team_2CC9AA36.csv         # ixtiyoriy (12-bo'lim solishtiruvi uchun)
```
**Tartib:** 1 → 13 ketma-ket (*Run all*). Birinchi kod hujayra `!pip -q install catboost` ni bajaradi (Colab'da lightgbm va pyarrow odatda bor).
**Taxminiy vaqt (Colab CPU, 2 yadro):** yuklash ~0.5 daq, featurelar ~1 daq, LR ~1–2 daq, LightGBM ~2–4 daq,
CatBoost ~5–10 daq, blend/yozish < 10 s → **jami ~10–20 daqiqa**. Xotira: ~3–4 GB.

**Eslatmalar:**
- `LR_TEST_MODE="full_fit"` (standart) — loyihadagi yakuniy fayl shu bilan olingan. `"fold_mean"` — muqobil (15 fold-model o'rtachasi).
- LGB/CAT natijalari kutubxona versiyasiga bog'liq bo'lishi mumkin; loyiha natijalari Colab'da olingan (versiyalar 7-bo'limda chiqariladi).